# Tuba v4 Elements, Sections, & 3D Supports Visualization Demo

This notebook demonstrates the newly ported features from **Tuba v2 into Tuba v4**, including:
1. **Section Profiles:** Pipes (`PipeSection`), solid/hollow bars (`BarSection`), cables (`CableSection`), rectangular/box beams (`RectangularSection`), and general I-beams (`IBeamSection` loaded from CSV profile database).
2. **Structural Elements:** Beams, bars, cables, and pipes.
3. **Advanced Supports:** Springs (stiffness matrices), concentrated masses, and custom blocked DOFs.
4. **Interactive 3D Visualizations:** Render the model and supports (Anchors as Red Cubes, Guides as Green Cylinders, Rests as Blue Cones, Springs as Yellow Spheres) directly in Jupyter.

In [ ]:
from __future__ import annotations
import sys
import tempfile
from pathlib import Path
import numpy as np
import pyvista as pv

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model, Material, PipingBuilder
from tuba.solver.aster import CodeAsterSolver
from tuba.visualizer import plots

# Enable interactive notebook rendering
pv.set_jupyter_backend("client")

## 1. Construct the Model with All Element & Section Types

We will construct a 3D piping system that includes all element types and section profiles:
- Pipe straight and bends (`PipeSec`)
- Solid/Hollow bar (`BarSec`)
- Tension-only cable (`CableSec`)
- Hollow box beam (`RectSec`)
- Standard I-beam (`IBeamSec` loaded from database)

In [ ]:
model = Model("Tuba_v4_Demo_Study")

# 1. Materials
model.add_material("Steel", E=2.0e11, nu=0.3, alpha=1.2e-5, rho=7850)

# 2. Sections
model.add_pipe_section("PipeSec", OD=0.1143, WT=0.006)
model.add_bar_section("BarSec", OD=0.05, WT=0.0)  # Solid bar
model.add_cable_section("CableSec", radius=0.01, pretension=500.0)
model.add_rectangular_section("RectSec", height_y=0.08, height_z=0.04, thickness_y=0.005, thickness_z=0.005)

# Load I-Beam from local database
try:
    model.add_ibeam_section("IBeamSec", "IPE100")
except Exception:
    # Fallback in case of custom run environment
    from tuba.model import IBeamSection
    model.sections["IBeamSec"] = IBeamSection(name="IBeamSec", profile_name="IPE100", properties={
        "A": 1.03e-3, "IY": 1.71e-6, "IZ": 1.59e-7, "JX": 1.2e-8, "RY": 4.07e-2, "RZ": 1.24e-2
    })

# 3. PipingBuilder - Routing elements
with model.pipe(section="PipeSec", material="Steel") as b:
    b.start([0, 0, 0], support="anchor")
    b.run(2.0)
    b.bend(radius=0.2, angle=90, plane="XY")
    b.run(1.5)

# Add beam, bar, cable, and rectangular elements
with model.pipe(section="IBeamSec", material="Steel") as b:
    b.start([2.2, 1.7, 0]).beam(1.5)

with model.pipe(section="BarSec", material="Steel") as b:
    b.start([3.7, 1.7, 0]).bar(1.0)

with model.pipe(section="CableSec", material="Steel") as b:
    b.start([4.7, 1.7, 0]).cable(2.0)

with model.pipe(section="RectSec", material="Steel") as b:
    b.start([6.7, 1.7, 0]).beam(1.2)

## 2. Define Advanced Supports

Now, we add advanced support configurations:
- **N1:** Spring support with translational stiffness in Y (`stiffness=1.5e6`).
- **N2:** Custom boundary condition with translations in X and Y blocked, rotation in Z blocked (`blocked_dof=[1, 1, 0, 0, 0, 1]`).
- **N3:** Spring matrix support with custom directional stiffnesses.
- **N4:** Rest support supporting a concentrated mass of 50 kg.

In [ ]:
# Node N0 is anchor (set during start)
# Let's add other supports
model.add_support(node="N1", type="spring", stiffness=1.5e6)
model.add_support(node="N2", type="custom", blocked_dof=[1, 1, 0, 0, 0, 1])
model.add_support(node="N3", type="spring", stiffness_matrix=[1e5, 2e5, 3e5, 0.0, 0.0, 0.0])
model.add_support(node="N4", type="rest", mass=50.0)

model.define_load_case("LoadCase1", gravity=True, pressure=1.0e6, temperature=120.0)

print("Supports in model:")
for s in model.supports:
    print(f"Node: {s.node:4} | Type: {s.type:8} | stiffness: {s.stiffness} | blocked_dof: {s.blocked_dof}")

## 3. Visualizing 3D Model Geometry and Supports

Let's use our new interactive PyVista plotting helpers to render the 3D model, the 3D obstacles (if any), and the 3D supports.
Supports are displayed as:
- **Red Cube:** Anchor
- **Green Cylinder:** Guide
- **Blue Cone:** Rest/Bearing
- **Yellow Sphere:** Spring
- **Magenta Sphere:** Custom/Other

In [ ]:
from tuba.visualizer.pipeline import build_mesh_from_model, inflate_tubes

# Construct PyVista line mesh
mesh = build_mesh_from_model(model)
radius = 0.05715 # tube inflation radius

# Inflate to 3D tubes
tubes = inflate_tubes(mesh, radius=radius)

# Render scene
p = pv.Plotter()
p.set_background("#1a1a2e")
p.add_text("Tuba v4 - Interactive Elements & Supports Scene", position="upper_left", font_size=12)

p.add_mesh(tubes, color="#5c6b73", opacity=0.85, label="Pipeline")

# Add supports using our new helper
plots._add_supports_to_plotter(p, model, scale=0.18)

p.add_legend(bcolor="#1a1a2e")
p.show()

## 4. Run Code_Aster Solving and Render FEA Results

If Code_Aster is set up in your WSL or Docker environment, we can solve and plot results (stress, deformation, reaction forces). If you only want to review the generated input files, we can call `solver.export_study()`.

In [ ]:
# Export the Code_Aster study files to a temp directory
solver = CodeAsterSolver()
tmp_dir = Path("./code_aster_export_demo")
solver.export_study(model, "LoadCase1", tmp_dir)
print(f"Code_Aster input files exported to: {tmp_dir.resolve()}")
print(f"Generated mail file: {(tmp_dir / 'study.mail').name}")
print(f"Generated comm file: {(tmp_dir / 'study.comm').name}")

### Running the solver and visualizing results:
```python
# To run the solver (requires WSL / Docker):
# results = model.solve(solver="code_aster")

# Once solved, you can display:
# 1. Stress distribution on 3D warped pipe:
#    results.plot_deformed_stress(deform_scale=50.0)
#
# 2. Reaction forces at supports (red arrows overlaying the support shapes):
#    results.plot_reactions()
#
# 3. Displacement vectors (cyan arrows):
#    results.plot_displacement_vectors()
```